# 01 — Data Collection & Feature Matrix Construction

This notebook loads raw data from the client Excel and O*NET database,
then builds the feature matrix used for clustering and network analysis.

**Owner**: Person A

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.onet_data import get_valid_codes, build_feature_matrix, load_related_occupations, load_job_zones
from src.data_cleaning import (
    load_agc_members, load_dot_prequal, load_apprenticeships,
    load_community_college, load_umaine, load_onet_codes
)

EXCEL = '../data/raw/Career_Drive_Project_Data_Sources.xlsx'
ONET_DB = '../data/raw/onet_db'

## 1. Extract valid O*NET codes from Excel

The Excel sheet has 28 rows but only 21 unique valid codes after filtering out
section headers, blanks, and duplicates. One code (13-1082.00) has no data in
the O*NET database, leaving us with 20 usable occupations.

In [2]:
codes = get_valid_codes(EXCEL)
print(f'Valid unique codes: {len(codes)}')
for c in codes:
    print(f'  {c}')

Valid unique codes: 21
  11-9021.00
  47-2031.00
  47-2111.00
  47-2221.00
  47-2152.00
  47-2081.00
  47-2132.00
  47-4021.00
  17-2051.00
  17-2051.01
  47-2071.00
  47-2073.00
  47-4051.00
  47-2151.00
  47-2072.00
  47-2171.00
  47-4061.00
  47-1011.00
  13-1082.00
  11-9041.00
  11-3013.00


## 2. Build feature matrix from O*NET database

In [3]:
features = build_feature_matrix(ONET_DB, codes)
print(f'Feature matrix: {features.shape}')
print(f'Missing values: {features.drop(columns="Title").isna().sum().sum()}')
features[['Title']].head(10)

Feature matrix: (20, 121)
Missing values: 0


,Title
O*NET-SOC Code,
11-3013.00,Facilities Managers
11-9021.00,Construction Managers
11-9041.00,Architectural and Engineering Managers
17-2051.00,Civil Engineers
17-2051.01,Transportation Engineers
47-1011.00,First-Line Supervisors of Construction Trades ...
47-2031.00,Carpenters
47-2071.00,"Paving, Surfacing, and Tamping Equipment Opera..."
47-2072.00,Pile Driver Operators


In [4]:
features.to_csv('../data/processed/occupation_features.csv')
print('Saved occupation_features.csv')

Saved occupation_features.csv


## 3. Load Related Occupations and Job Zones

In [5]:
# Related occupations (edges for network graph)
edges = load_related_occupations(ONET_DB, list(features.index))
print(f'Related occupation edges: {len(edges)}')
edges.to_csv('../data/processed/related_occupations.csv', index=False)

# Job zones (complexity labels for validation)
jz = load_job_zones(ONET_DB, list(features.index))
print(f'Job zones: {len(jz)}')
jz.to_csv('../data/processed/job_zones.csv', index=False)
print(jz.to_string(index=False))

Related occupation edges: 82
Job zones: 20
O*NET-SOC Code  Job Zone
    11-3013.00         3
    11-9021.00         4
    11-9041.00         5
    17-2051.00         4
    17-2051.01         4
    47-1011.00         3
    47-2031.00         2
    47-2071.00         2
    47-2072.00         2
    47-2073.00         2
    47-2081.00         2
    47-2111.00         3
    47-2132.00         2
    47-2151.00         2
    47-2152.00         3
    47-2171.00         2
    47-2221.00         2
    47-4021.00         3
    47-4051.00         2
    47-4061.00         2


## 4. Load client Excel data

In [6]:
agc = load_agc_members(EXCEL)
print(f'AGC Members: {len(agc)}')
print(agc['type'].value_counts())

AGC Members: 222
type
Specialty Contractors    67
General Contractors      60
Service Providers        58
Suppliers                35
Developer                 2
Name: count, dtype: int64


In [7]:
dot = load_dot_prequal(EXCEL)
print(f'DOT Prequal: {len(dot)}')

DOT Prequal: 57


In [8]:
apprentice = load_apprenticeships(EXCEL)
print(f'Apprenticeships: {len(apprentice)}')
apprentice.head()

Apprenticeships: 19


,onet_code,title,term_hours
0,595,Arborist,6000
1,90380,Bridge Carpenter/Heavy Highway,4000
2,90415,Construction Carpenter,5000
3,90708,Construction Craft Concrete Laborer,2000
4,90699,Construction Craft Heavy / Highway Laborer,2000


In [9]:
cc = load_community_college(EXCEL)
umaine = load_umaine(EXCEL)
print(f'Community College programs: {len(cc)}')
print(f'UMaine programs: {len(umaine)}')

Community College programs: 21
UMaine programs: 10


## 5. Save all cleaned datasets

In [10]:
agc.to_csv('../data/processed/agc_members.csv', index=False)
dot.to_csv('../data/processed/dot_prequal.csv', index=False)
apprentice.to_csv('../data/processed/apprenticeships.csv', index=False)
cc.to_csv('../data/processed/community_college.csv', index=False)
umaine.to_csv('../data/processed/umaine_programs.csv', index=False)
print('All datasets saved to data/processed/')

All datasets saved to data/processed/
